# Trening rdzenia GTO (Colab, POKER-69)

Runtime: **CPU, High-RAM**. GPU wyłączone — nie liczy artefaktu.
Gałąź: `grok/poker-53-aivat`. Profil `smoke` najpierw, `wta25` dopiero po zielonym dymie.
Sesja pada ~12 h. Bezpiecznik `--session-hours 9` (fuse działa po kompletnym cyklu/warstwie).
OUT na Drive. Artefaktu nie wrzucaj do publicznego repo (decyzja 30).
Tożsamość katalogu PROD: `colab_run.py identity --dir ...` — nie wołaj na OUT solvera.

In [ ]:
# 1. środowisko — zmienne idą do os.environ, bo ! widzi shell, nie Pythona
import os, sys
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
REPO = "/content/Poker"
if not os.path.isdir(REPO):
    %cd /content
    !git clone --branch grok/poker-53-aivat https://github.com/mcz91/Poker.git
    !pip install -e "/content/Poker[train]"
from google.colab import drive
drive.mount("/content/drive")
PROFILE = "smoke"  # potem: tdeep | wta25
os.environ["POKER_REPO"] = REPO
os.environ["POKER_PROFILE"] = PROFILE
os.environ["POKER_TENSOR"] = f"{REPO}/tools/blueprint/control/tensor" if PROFILE == "smoke" else "/content/drive/MyDrive/poker-gto/PROD/tensor"
os.environ["POKER_OUT"] = f"/content/drive/MyDrive/poker-gto/{PROFILE}"
os.environ["POKER_JOBS"] = str(os.cpu_count() or 2)
print({k: os.environ[k] for k in ("POKER_PROFILE", "POKER_TENSOR", "POKER_OUT", "POKER_JOBS")})

In [ ]:
# 2. solve — po restarcie odpal TĘ SAMĄ komórkę; --allow-fresh tylko gdy brak manifestu
import os
fresh = "--allow-fresh" if not os.path.exists(os.path.join(os.environ["POKER_OUT"], "solve_manifest.json")) else ""
hours = "0.25" if os.environ["POKER_PROFILE"] == "smoke" else "9"
!python $POKER_REPO/tools/blueprint/colab_run.py solve --profile $POKER_PROFILE --tensor $POKER_TENSOR --out $POKER_OUT --session-hours {hours} --jobs $POKER_JOBS {fresh}
!python $POKER_REPO/tools/blueprint/colab_run.py status --out $POKER_OUT

In [ ]:
# 3. pack tylko gdy status=done
!python $POKER_REPO/tools/blueprint/colab_run.py pack --run $POKER_OUT --bpk $POKER_OUT/blueprint_v2.bpk